In [1]:
import gdsfactory as gf


from mesalab_pdk import get_pdk, LAYER

pdk = get_pdk()
pdk.activate()

In [2]:

from scipy.optimize import minimize
import numpy as np

import sys, os
sys.path.append("/opt/lumerical/v252/api/python/")

import lumapi 

In [3]:

# 2. Define your SiN 300nm Component
@gf.cell
def sin_mmi_1x2(length=35.0, width=7.0, taper_w=1.8, taper_l=12.0):
    c = gf.Component()
    # gdsfactory standard 1x2 MMI builder
    mmi = c << gf.components.mmi1x2(
        length_mmi=length,
        width_mmi=width,
        width_taper=taper_w,
        length_taper=taper_l,
    )
    c.add_ports(mmi.ports)
    return c

def run_eme_sim(params):
    length, width = params
    
    # 1. Geometry Compilation
    c = sin_mmi_1x2(length=length, width=width)
    gds_path = os.path.abspath("temp_sin_mmi.gds")
    c.write_gds(gds_path)
    cell_name = c.name 
    
    with lumapi.MODE(hide=True) as mode:
        mode.new()
        
        # 2. Poly-Extrusion Imports
        mode.gdsimport(
            gds_path, 
            cell_name, 
            1,                                   
            "Si3N4 (Silicon Nitride) - Phillip", 
            -0.15e-6,                            
            0.15e-6                              
        )
        
        # 3. Solver Properties Instantiation
        mode.addeme()
        mode.set("solver type", "3D")
        mode.set("background material", "SiO2 (Glass) - Palik")
        mode.set("y span", (width + 4.0) * 1e-6)
        mode.set("z span", 3.0e-6)   
        mode.set("wavelength", 1.55e-6)
        
        # --- FIXED CONVERGENCE SETTINGS ---
        mode.set("number of cell groups", 3)
        mode.set("allow custom eigensolver settings", 0) # Disables lock
        mode.set("number of modes for all cell groups", 35) # High mode count for complex beating
        
        # Set spatial cell splits per group
        mode.set("group spans", np.array([[12.0e-6], [length * 1e-6], [12.0e-6]]))
        mode.set("cells", np.array([[10], [1], [10]])) 
        
        # 4. Engine Run Phase
        mode.run()
        
        # 5. Analysis Propagate Phase
        mode.setemeanalysis("group spans", np.array([[12.0e-6], [length * 1e-6], [12.0e-6]]))
        mode.emepropagate()
        
        # 6. S-Parameter Matrix Reduction
        s_matrix = mode.getresult("eme", "internal s matrix")
        transmission = np.abs(s_matrix)**2 + np.abs(s_matrix)**2
        
    if os.path.exists(gds_path):
        os.remove(gds_path)
        
    return -transmission
# 4. Run the Nelder-Mead Optimization Routine
# Initial Guess: Length = 40 um, Width = 7 um
initial_geometry = [40.0, 7.0] 
bounds = ((15.0, 80.0), (5.0, 12.0)) # Prevent algorithm from choosing physical extremes

res = minimize(run_eme_sim, initial_geometry, method='Nelder-Mead', bounds=bounds, options={'xatol': 0.1})

print(f"Optimization complete!")
print(f"Optimized MMI Length: {res.x[0]:.2f} um")
print(f"Optimized MMI Width:  {res.x[1]:.2f} um")


KeyboardInterrupt: 